In [ ]:
# structural_break_analysis.py
# 입력 파일: ./merged_var_input.csv

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from statsmodels.stats.diagnostic import breaks_cusumolsresid, breaks_hansen

warnings.filterwarnings("ignore")

# =========================
# 0) Config
# =========================
INPUT_CSV = "./merged_var_input.csv"
OUT_DIR = "./structural_break_out"
os.makedirs(OUT_DIR, exist_ok=True)

DATE_COL = "Date"

# 기본 회귀식:
# dlog_COPPER_t = alpha + beta*dlog_SOLVPN_t + controls + error
Y_COL = "dlog_COPPER"
X_COL = "dlog_SOLVPN"
CONTROL_COLS = ["dlog_DXY", "d_UST10Y", "dlog_VIX"]

MIN_SEG = 80
START_FRAC = 0.15
END_FRAC = 0.85

# =========================
# 1) Load
# =========================
df = pd.read_csv(INPUT_CSV)
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL).reset_index(drop=True)

use_cols = [DATE_COL, Y_COL, X_COL] + CONTROL_COLS
df = df[use_cols].dropna().reset_index(drop=True)

print("Loaded shape:", df.shape)
print("Date range:", df[DATE_COL].min(), "~", df[DATE_COL].max())

# =========================
# 2) Helpers
# =========================
def make_design(data):
    X = data[[X_COL] + CONTROL_COLS].copy()
    X = sm.add_constant(X, has_constant="add")
    y = data[Y_COL].astype(float).copy()
    return y, X

def fit_ols(data):
    y, X = make_design(data)
    return sm.OLS(y, X).fit()

def chow_f_stat(full_df, break_idx):
    left = full_df.iloc[:break_idx].copy()
    right = full_df.iloc[break_idx:].copy()

    n1, n2 = len(left), len(right)
    if n1 <= MIN_SEG or n2 <= MIN_SEG:
        return np.nan

    m_pool = fit_ols(full_df)
    m1 = fit_ols(left)
    m2 = fit_ols(right)

    rss_p = np.sum(m_pool.resid ** 2)
    rss_1 = np.sum(m1.resid ** 2)
    rss_2 = np.sum(m2.resid ** 2)

    k = int(m_pool.df_model + 1)
    denom_df = n1 + n2 - 2 * k
    if denom_df <= 0:
        return np.nan

    num = (rss_p - (rss_1 + rss_2)) / k
    den = (rss_1 + rss_2) / denom_df
    if den <= 0:
        return np.nan

    return num / den

# =========================
# 3) Full OLS
# =========================
full_model = fit_ols(df)

with open(os.path.join(OUT_DIR, "full_ols_summary.txt"), "w", encoding="utf-8") as f:
    f.write(str(full_model.summary()))

# =========================
# 4) Stability tests
# =========================
cusum_stat, cusum_pvalue, cusum_crit = breaks_cusumolsresid(
    full_model.resid,
    ddof=int(full_model.df_model + 1)
)
hansen_stat = breaks_hansen(full_model)

with open(os.path.join(OUT_DIR, "stability_test_summary.txt"), "w", encoding="utf-8") as f:
    f.write(f"CUSUM stat: {cusum_stat}\n")
    f.write(f"CUSUM p-value: {cusum_pvalue}\n")
    f.write(f"CUSUM crit: {cusum_crit}\n\n")
    f.write(f"Hansen test: {hansen_stat}\n")

print("CUSUM stat:", cusum_stat)
print("CUSUM p-value:", cusum_pvalue)
print("Hansen test:", hansen_stat)

# =========================
# 5) Single break scan
# =========================
n = len(df)
start = max(MIN_SEG, int(n * START_FRAC))
end = min(n - MIN_SEG, int(n * END_FRAC))

rows = []
for b in range(start, end):
    fval = chow_f_stat(df, b)
    rows.append({
        "break_idx": b,
        "break_date": df.iloc[b][DATE_COL],
        "F_stat": fval
    })

scan_df = pd.DataFrame(rows).dropna().reset_index(drop=True)
scan_df.to_csv(os.path.join(OUT_DIR, "break_scan_full.csv"), index=False, encoding="utf-8-sig")

best_single = scan_df.sort_values("F_stat", ascending=False).head(1).copy()
best_single.to_csv(os.path.join(OUT_DIR, "best_single_break.csv"), index=False, encoding="utf-8-sig")

print("\nBest single break:")
print(best_single)

# =========================
# 6) Plot: break scan
# =========================
plt.figure(figsize=(14, 5))
plt.plot(scan_df["break_date"], scan_df["F_stat"])
plt.title(f"Structural Break Scan: {Y_COL} ~ {X_COL} + controls")
plt.xlabel("Date")
plt.ylabel("F-stat")
plt.xticks(rotation=30)

if len(best_single) > 0:
    bd = pd.to_datetime(best_single.iloc[0]["break_date"])
    plt.axvline(bd, linestyle="--")
    plt.text(bd, scan_df["F_stat"].max() * 0.95, f"{bd.date()}")

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "break_scan_plot.png"), dpi=300)
plt.close()

# =========================
# 7) Rolling beta
# =========================
window = 120
beta_rows = []

for end_idx in range(window, len(df) + 1):
    sub = df.iloc[end_idx - window:end_idx].copy()
    m = fit_ols(sub)
    beta_rows.append({
        DATE_COL: sub.iloc[-1][DATE_COL],
        "rolling_beta": m.params.get(X_COL, np.nan)
    })

roll_beta_df = pd.DataFrame(beta_rows)
roll_beta_df.to_csv(os.path.join(OUT_DIR, "rolling_beta_120.csv"), index=False, encoding="utf-8-sig")

plt.figure(figsize=(14, 5))
plt.plot(roll_beta_df[DATE_COL], roll_beta_df["rolling_beta"])
plt.axhline(0.0, linestyle="--")
if len(best_single) > 0:
    plt.axvline(pd.to_datetime(best_single.iloc[0]["break_date"]), linestyle="--")

plt.title(f"Rolling Beta (window=120): beta of {X_COL} in {Y_COL} regression")
plt.xlabel("Date")
plt.ylabel("Rolling beta")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "rolling_beta_plot.png"), dpi=300)
plt.close()

print("\nSaved to:", OUT_DIR)

Loaded shape: (1325, 6)
Date range: 2020-10-13 00:00:00 ~ 2026-01-12 00:00:00
CUSUM stat: 0.6466514879223865
CUSUM p-value: 0.7971795496009251
Hansen test: (np.float64(2.220086159881003), array([( 2, 1.01), ( 6, 1.9 ), (15, 3.75), (19, 4.52)],
      dtype=[('nobs', '<i8'), ('crit', '<f8')]))

Best single break:
     break_idx break_date    F_stat
927       1125 2025-03-27  4.159765

Saved to: ./structural_break_out


: 